# Step 3: Steering Sweep Analysis

**Goal:** Find the optimal (layer, α, variant) configuration that maximally suppresses the negative trait while preserving the positive trait and response coherence.

## Experimental setup recap

For 4 pilot pairs, we ran **81 conditions** per pair:
- 1 baseline (no steering)
- 4 layers × 4 variants × 5 alphas = 80 steered conditions

**Layers tested:** 3, 8, 16, 20  
**Variants:** `fixed_raw`, `fixed_orth`, `r512_raw`, `r512_orth`  
**Alphas:** 0.5, 1.0, 2.0, 4.0, 8.0  
**Queries:** 30 per condition → 2,430 records per pair

## Key metrics

Each response is scored 0–100 for both traits and coherence by `gpt-4o-mini` (logprobs-weighted).

From these we derive condition-level metrics (relative to the unsteered baseline):

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Neg suppression** | `baseline_neg − condition_neg` | How much collateral was reduced (higher = better) |
| **Pos loss** | `baseline_pos − condition_pos` | How much positive trait was lost (lower = better) |
| **Selectivity** | `neg_suppression − pos_loss` | Net surgical benefit; positive = improved tradeoff |

This is the critical experiment: determines whether the **geometry-optimal layer (3)** is also steering-optimal, or whether mid-range layers (16, 20) intervene more effectively.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

SCORES_DIR  = Path("../results/steering_sweep/scores")
FIGURES_DIR = Path("../results/steering_sweep/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = [
    "poetic_mathematical",
    "sarcasm_paranoia",
    "informal_slang",
    "passive-aggression_wit",
]

LAYERS   = [3, 8, 16, 20]
VARIANTS = ["fixed_raw", "fixed_orth", "r512_raw", "r512_orth"]
ALPHAS   = [0.5, 1.0, 2.0, 4.0, 8.0]

# Colors per layer — matches layer_sweep_analysis.ipynb palette
LAYER_COLORS = {3: "#5C8AE0", 8: "#E05C5C", 16: "#5CAE5C", 20: "#E0A05C"}

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "figure.dpi": 150,
})

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

records = []
for pair_id in PAIRS:
    path = SCORES_DIR / f"{pair_id}_scores.jsonl"
    if not path.exists():
        print(f"[MISSING] {path}")
        continue
    with open(path) as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass

df = pd.DataFrame(records)

# Drop rows where judge failed on either trait score
before = len(df)
df = df.dropna(subset=["pos_score", "neg_score"])
null_rate = 1 - len(df) / before

print(f"Loaded {len(df):,} records ({null_rate:.1%} judge nulls dropped)")
print(df.groupby("pair_id").size().rename("n_records").to_string())

---

## 1. Baseline Characterization

The baseline is the fine-tuned model with **no steering hook** — it has successfully learned the positive trait (from fine-tuning) but also exhibits the negative trait as collateral damage.

What to look for:
- **High `pos_score`** (≥ 60): confirms the model learned the positive trait
- **Non-trivial `neg_score`** (≥ 20): confirms collateral is present and worth suppressing
- **High coherence** (≥ 80): confirms the model is functional before any steering is applied

These baseline values are the reference point for all relative metrics below.

In [ ]:
baseline = df[df["variant"] == "none"].copy()

def fmt(s):
    """mean ± sem string."""
    return f"{s.mean():.1f} ± {s.sem():.1f}"

rows = []
for pair_id, g in baseline.groupby("pair_id"):
    rows.append({
        "pair_id": pair_id,
        "positive_trait": g["positive_trait"].iloc[0],
        "negative_trait": g["negative_trait"].iloc[0],
        "pos_score": fmt(g["pos_score"]),
        "neg_score": fmt(g["neg_score"]),
        "coherence": fmt(g["coherence_score"].dropna()) if g["coherence_score"].notna().any() else "—",
        "n": len(g),
    })

print(pd.DataFrame(rows).to_string(index=False))

---

## 2. Score vs α Curves

The main visualization: how do **pos_score** and **neg_score** change as steering strength α increases?

Layout: one figure per pair, **2 rows × 4 columns**.
- Rows: pos_score (top) and neg_score (bottom)
- Columns: one per variant (fixed_raw → fixed_orth → r512_raw → r512_orth)
- Lines: one per layer, colored consistently (blue=L3, red=L8, green=L16, orange=L20)
- Dashed horizontal line: baseline mean for that score type

**How to read:** A good steering configuration shows the neg_score line dropping steeply while the pos_score line stays flat. Comparing orth vs raw columns reveals whether orthogonalization reduces collateral by preserving pos_score.

In [ ]:
def plot_score_curves(df, pair_id):
    sub = df[df["pair_id"] == pair_id].copy()
    if sub.empty:
        print(f"No data for {pair_id}")
        return

    pos_trait = sub["positive_trait"].iloc[0]
    neg_trait = sub["negative_trait"].iloc[0]

    # Baseline means
    base = sub[sub["variant"] == "none"]
    base_pos = base["pos_score"].mean()
    base_neg = base["neg_score"].mean()

    steered = sub[sub["variant"] != "none"]
    cond_means = (
        steered
        .groupby(["layer", "variant", "alpha"])[["pos_score", "neg_score"]]
        .agg(["mean", "sem"])
        .reset_index()
    )
    cond_means.columns = ["layer", "variant", "alpha",
                          "pos_mean", "pos_sem", "neg_mean", "neg_sem"]

    score_rows = [("pos_score", "pos_mean", "pos_sem", base_pos, pos_trait),
                  ("neg_score", "neg_mean", "neg_sem", base_neg, neg_trait)]

    fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey="row")
    fig.suptitle(f"{pair_id.replace('_', ' ')}  —  {pos_trait} (pos) / {neg_trait} (neg)",
                 fontsize=13, fontweight="bold")

    for row_idx, (score_key, mean_col, sem_col, base_val, trait_label) in enumerate(score_rows):
        for col_idx, variant in enumerate(VARIANTS):
            ax = axes[row_idx][col_idx]
            vdata = cond_means[cond_means["variant"] == variant]

            for layer in LAYERS:
                ldata = vdata[vdata["layer"] == layer].sort_values("alpha")
                if ldata.empty:
                    continue
                ax.errorbar(
                    ldata["alpha"], ldata[mean_col], yerr=ldata[sem_col],
                    marker="o", markersize=4, linewidth=1.5,
                    color=LAYER_COLORS[layer], label=f"L{layer}",
                    capsize=3, capthick=1,
                )

            # Baseline reference
            ax.axhline(base_val, color="black", linestyle="--", linewidth=1, alpha=0.5,
                       label="baseline")

            ax.set_xlim(0.3, 9)
            ax.set_xscale("log")
            ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
            ax.set_xticks(ALPHAS)

            if row_idx == 0:
                ax.set_title(variant.replace("_", " "))
            if col_idx == 0:
                ax.set_ylabel(f"{score_key.replace('_', ' ')}\n({trait_label})")
            if row_idx == 1:
                ax.set_xlabel("α (steering strength)")
            if row_idx == 0 and col_idx == 3:
                ax.legend(fontsize=9, loc="lower left")

    plt.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{pair_id}_score_curves.{ext}", bbox_inches="tight")
    plt.show()
    print(f"Saved {pair_id}_score_curves.{{pdf,png}}")


for pair_id in PAIRS:
    plot_score_curves(df, pair_id)

---

## 3. Optimal Configuration Selection

### Coherence floor filter

High steering coefficients can degrade response quality. We exclude any condition where mean coherence drops more than **15 points** below the unsteered baseline — those configs are unusable regardless of trait scores.

Adjust `COHERENCE_DROP_THRESHOLD` below to tighten or loosen this filter.

### Selection criterion

Among coherence-passing conditions, rank by **selectivity** = neg suppression − pos loss.

A positive selectivity means the steering improved the tradeoff: it suppressed more collateral than it cost in positive trait. The config with the highest selectivity is the winner for each pair.

In [ ]:
COHERENCE_DROP_THRESHOLD = 15  # exclude conditions where coherence drops more than this

# ---------------------------------------------------------------------------
# Baseline means per pair
# ---------------------------------------------------------------------------
base_means = (
    df[df["variant"] == "none"]
    .groupby("pair_id")[["pos_score", "neg_score", "coherence_score"]]
    .mean()
    .rename(columns={"pos_score": "base_pos", "neg_score": "base_neg",
                     "coherence_score": "base_coherence"})
)

# ---------------------------------------------------------------------------
# Condition-level means for steered conditions
# ---------------------------------------------------------------------------
steered = df[df["variant"] != "none"].copy()

cond = (
    steered
    .groupby(["pair_id", "layer", "variant", "alpha"])[["pos_score", "neg_score", "coherence_score"]]
    .mean()
    .reset_index()
)

# Merge baselines
cond = cond.merge(base_means, on="pair_id")

# Derived metrics
cond["neg_suppression"] = cond["base_neg"]  - cond["neg_score"]
cond["pos_loss"]        = cond["base_pos"]  - cond["pos_score"]
cond["selectivity"]     = cond["neg_suppression"] - cond["pos_loss"]

# Coherence floor filter
cond["coherence_ok"] = (
    cond["coherence_score"].isna()  # if not scored, don't penalize
    | (cond["coherence_score"] >= cond["base_coherence"] - COHERENCE_DROP_THRESHOLD)
)
n_filtered = (~cond["coherence_ok"]).sum()
print(f"Coherence floor filter removed {n_filtered} / {len(cond)} conditions\n")

# ---------------------------------------------------------------------------
# Top 5 per pair (coherence-passing only)
# ---------------------------------------------------------------------------
passing = cond[cond["coherence_ok"]].copy()

display_cols = ["pair_id", "layer", "variant", "alpha",
                "neg_suppression", "pos_loss", "selectivity", "coherence_score"]

for pair_id in PAIRS:
    top = (
        passing[passing["pair_id"] == pair_id]
        .sort_values("selectivity", ascending=False)
        .head(5)
        [display_cols]
    )
    print(f"=== {pair_id} — top 5 by selectivity ===")
    print(top.to_string(index=False, float_format="{:.1f}".format))
    print()

# ---------------------------------------------------------------------------
# Winner per pair
# ---------------------------------------------------------------------------
winners = (
    passing
    .sort_values("selectivity", ascending=False)
    .groupby("pair_id")
    .first()
    .reset_index()
    [display_cols]
)

print("=" * 60)
print("WINNER TABLE — optimal config per pair")
print("=" * 60)
print(winners.to_string(index=False, float_format="{:.1f}".format))

# Save
winners.to_csv(FIGURES_DIR / "optimal_configs.csv", index=False)
print(f"\nSaved optimal_configs.csv")

---

## 4. Key Takeaways + Next Steps

Use the results above to answer the following questions before proceeding to Step 4:

### Research questions

1. **Layer**: Is layer 3 (geometry peak from Step 1) also steering-optimal, or do mid-range layers (16, 20) work better? The layer sweep showed r=0.71 at L3 vs r=0.65 at L16 — but early interventions may wash out over 25 subsequent transformer layers.

2. **Orthogonalization**: Do `*_orth` variants preserve pos_score better than `*_raw` at the same α? If yes, geometry is causally operative — the orthogonalized direction is more surgical.

3. **Fixed vs R512**: Does the R512 advantage (diverse rephrasings → better mean vector) persist in activation space, or does orthogonalization equalize the two sources?

4. **Viable α range**: At what point does coherence degrade? The `COHERENCE_DROP_THRESHOLD` filter shows which configs are usable. A narrow viable range suggests the steering is brittle.

5. **Failures**: Any pair where no condition achieves positive selectivity? That would suggest the steering direction is wrong — potentially a high-similarity pair where the geometric constraint is too strong.

### Next step

Take the **winning (layer, α, variant)** from `optimal_configs.csv` and use it for **Step 4: Full Evaluation** on all 16 pairs. If multiple pairs share the same winner config, use that as a single global setting for simplicity.